In [ ]:
# NB00b_gold_config  --  shared constants and helpers for NB02, NB03 and NB04
# Run after NB00_setup in the same session (%run NB00_setup, then %run NB00b_gold_config)
import time, datetime
from pyspark.sql import functions as F, Window

SRC_TELEMETRY = "silver_telemetry_anomalies"   # one row per reading + vehicle enrichment + flags
SRC_EPISODES  = "silver_anomaly_episodes"

MAX_GAP_SEC             = 300     # a gap above 5 minutes is not a real interval (e.g. between two trips)
MAX_PLAUSIBLE_SPEED_KMH = 200.0   # above this = GPS teleport, distance is not counted
SPEEDING_KMH            = 90.0
COLD_MIN, COLD_MAX      = 2.0, 8.0

def flag(c):
    return F.coalesce(F.col(c), F.lit(False))

valid_iv = (F.col("time_gap_sec").isNotNull() & (F.col("time_gap_sec") > 0)
            & (F.col("time_gap_sec") <= MAX_GAP_SEC))
ok_dist = (valid_iv & (~flag("is_gps_teleportation"))
           & (F.coalesce(F.col("calculated_speed_kmh"), F.lit(0.0)) <= MAX_PLAUSIBLE_SPEED_KMH))

# ---- measure definitions (the same definitions are used in the facts and in the silver-side reconciliation) ----
distance_expr = F.when(ok_dist, F.col("distance_km")).otherwise(F.lit(0.0))
# negative fuel_rate (-5.0) = sensor error value, so fuel for this interval is counted as 0
fuel_expr     = F.when(valid_iv & (F.col("fuel_rate_lph") >= 0),
                       F.col("fuel_rate_lph") * F.col("time_gap_sec") / 3600.0).otherwise(F.lit(0.0))
idle_min_expr = F.when(valid_iv & flag("is_idling"), F.col("time_gap_sec") / 60.0).otherwise(F.lit(0.0))
is_refrig     = (F.col("vehicle_type") == "refrigerated_truck")
cold_total_expr = F.when(is_refrig & F.col("cargo_temp_c").isNotNull(), F.lit(1)).otherwise(F.lit(0))
cold_ok_expr    = F.when(is_refrig & F.col("cargo_temp_c").between(COLD_MIN, COLD_MAX), F.lit(1)).otherwise(F.lit(0))
speeding_expr   = F.when(F.col("speed_kmh") > SPEEDING_KMH, F.lit(1)).otherwise(F.lit(0))


def add_corridor(df, has_trip=True):
    """Add corridor_id: from trip_id when it exists in ref_trips, otherwise the corridor the vehicle uses most often."""
    vr = (spark.table("ref_trips").groupBy("vehicle_id", "corridor_id").agg(F.count("*").alias("n"))
          .withColumn("rn", F.row_number().over(
              Window.partitionBy("vehicle_id").orderBy(F.desc("n"), F.asc("corridor_id"))))
          .where("rn = 1").select("vehicle_id", F.col("corridor_id").alias("veh_corridor_id")))
    out = df.join(F.broadcast(vr), "vehicle_id", "left")
    if has_trip:
        tm = (spark.table("ref_trips")
              .select("trip_id", F.col("corridor_id").alias("trip_corridor_id"))
              .dropDuplicates(["trip_id"]))
        out = out.join(F.broadcast(tm), "trip_id", "left")
        return (out.withColumn("corridor_id", F.coalesce("trip_corridor_id", "veh_corridor_id"))
                   .drop("trip_corridor_id", "veh_corridor_id"))
    return out.withColumn("corridor_id", F.col("veh_corridor_id")).drop("veh_corridor_id")


def dedupe_telemetry(df):
    """silver has repeated event_id values. Keep one row per event_id, preferring the row with the largest (valid) time gap."""
    w = Window.partitionBy("event_id").orderBy(F.col("time_gap_sec").desc_nulls_last(),
                                               F.col("distance_km").desc_nulls_last(),
                                               F.col("speed_kmh").asc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


def log_run(job, t0, rows=None):
    """Log the run time of each job into gold_job_run_times (used for the performance record)."""
    (spark.createDataFrame([(job, datetime.datetime.utcnow(), round(time.time() - t0, 1), rows)],
                           "job string, finished_at timestamp, seconds double, rows long")
          .write.mode("append").format("delta").saveAsTable("gold_job_run_times"))
    print(f"[run time] {job}: {round(time.time() - t0, 1)} s")

print("NB00b ready ✔")